# Serie anual de deforestación por departamento — SMByC

Procesa los anexos oficiales de cifras del Sistema de Monitoreo de Bosques y Carbono (IDEAM):
serie anual de deforestación con desagregación departamental, periodo 2013-2024.

**Fuente:** anexos de cifras del monitoreo de bosque del SMByC, servidos desde el portal
principal del IDEAM (`ideam.gov.co`). Ver detalle de la ruta y las URLs en el `README.md` de
esta carpeta.


In [11]:
import re
from pathlib import Path

import pandas as pd
import plotly.graph_objects as go

CARPETA = Path.cwd()
ARCHIVO_DPTO = CARPETA / "4.02_d_cambio_bosque_dpto.xlsx"
ARCHIVO_NACIONAL = CARPETA / "4.02_d_cambio_bosque_nacional.xlsx"

DEPARTAMENTOS_AMAZONICOS = [
    "AMAZONAS", "CAQUETÁ", "GUAINÍA", "GUAVIARE", "PUTUMAYO", "VAUPÉS",
]


In [12]:
# Hojas con periodicidad anual, según la nota metodológica del IDEAM (fila 52 de cada hoja):
# solo 2012-2013 en adelante tienen periodicidad anual; 1990-2000, 2000-2005, 2005-2010 y
# 2010-2012 son periodos multianuales y quedan fuera de la serie anual.
xl = pd.ExcelFile(ARCHIVO_DPTO)
hojas_anuales = sorted(
    hoja for hoja in xl.sheet_names
    if re.fullmatch(r"20\d{2}-20\d{2}", hoja) and int(hoja[:4]) >= 2012
)

series_por_anio = {}
for hoja in hojas_anuales:
    anio = int(hoja.split("-")[1])  # año final del periodo (ej. 2012-2013 -> 2013)
    hoja_df = pd.read_excel(ARCHIVO_DPTO, sheet_name=hoja, header=None)

    # El header ("Departamento", "Superficie deforestada SD", ...) cambia de fila entre
    # hojas; se ubica buscando la celda "Departamento" en la columna 1.
    fila_header = next(
        i for i in range(len(hoja_df)) if str(hoja_df.iloc[i, 1]).strip() == "Departamento"
    )
    datos = hoja_df.iloc[fila_header + 1:, [1, 2]].copy()
    datos.columns = ["departamento", "sd"]
    datos["departamento"] = datos["departamento"].astype(str).str.strip()
    datos = datos[datos["departamento"].notna() & (datos["departamento"] != "nan")]
    datos = datos[~datos["departamento"].str.lower().eq("total")]  # excluir fila de totales
    datos = datos.dropna(subset=["sd"])
    datos["sd"] = datos["sd"].astype(float)

    series_por_anio[anio] = datos.set_index("departamento")["sd"]

# Departamentos como filas, años como columnas, hectáreas deforestadas (SD) como valores.
df_dpto = pd.DataFrame(series_por_anio).sort_index(axis=1)
df_dpto.head()


,2013,2014,2015,2016,2017,2018,2019,2020,2021,2022,2023,2024
departamento,,,,,,,,,,,,
AMAZONAS,1042.0,1723.0,1277.0,1913.0,1362.0,782.0,1139.0,2669.0,962.0,1157.0,1862.056120,1782.669373
ANTIOQUIA,13736.0,21032.0,15888.0,20494.0,20592.0,12820.0,11601.0,12645.0,9751.0,10290.0,8138.721853,7206.996603
ARAUCA,3119.0,3913.0,2395.0,3907.0,3214.0,4204.0,3452.0,2175.0,2537.0,2189.0,1583.272919,1147.575398
ATLÁNTICO,62.0,55.0,40.0,72.0,50.0,0.0,18.0,26.0,13.0,27.0,4.647936,43.318764
"BOGOTÁ,D.C",1.0,0.0,2.0,1.0,28.0,11.0,0.0,0.0,0.0,0.0,0.000000,0.000000


In [13]:
# Serie nacional oficial (no la suma de departamentos): el IDEAM advierte que el total
# nacional y la suma departamental pueden diferir por redondeo y por áreas sin información
# que no aplican de igual forma a nivel departamental.
nac_df = pd.read_excel(ARCHIVO_NACIONAL, sheet_name="Nacional", header=None)
fila_header_nac = next(
    i for i in range(len(nac_df)) if str(nac_df.iloc[i, 1]).strip().startswith("Periodo")
)
datos_nac = nac_df.iloc[fila_header_nac + 1:, [1, 2]].copy()
datos_nac.columns = ["periodo", "sd"]
datos_nac = datos_nac.dropna(subset=["periodo"])

serie_nacional = {}
for _, fila in datos_nac.iterrows():
    anios = re.findall(r"20\d{2}|19\d{2}", str(fila["periodo"]))
    if len(anios) != 2:
        continue
    ini, fin = int(anios[0]), int(anios[1])
    if fin - ini == 1 and ini >= 2012:  # solo periodos anuales, 2012-2013 en adelante
        serie_nacional[fin] = float(fila["sd"])

serie_nacional = pd.Series(serie_nacional).sort_index()
serie_nacional


2013    120938.000000
2014    140356.000000
2015    124035.000000
2016    178597.000000
2017    219973.000000
2018    197159.000000
2019    158894.000000
2020    171685.000000
2021    174103.000000
2022    123517.411860
2023     79256.325796
2024    113607.919350
dtype: float64

In [14]:
# Departamentos amazónicos: suma de su deforestación anual (no viene desagregada en el
# anexo oficial, se construye a partir de la tabla departamental).
serie_amazonia = df_dpto.loc[df_dpto.index.isin(DEPARTAMENTOS_AMAZONICOS)].sum(axis=0)

# Departamentos con mayor deforestación promedio en el periodo, para la gráfica.
top_departamentos = df_dpto.loc[df_dpto.mean(axis=1).nlargest(6).index]

salida_csv = CARPETA / "deforestacion_departamentos.csv"
df_dpto.to_csv(salida_csv, encoding="utf-8-sig")
print(f"Tabla exportada en: {salida_csv}")

print()
print("Total nacional oficial por año (ha):")
print(serie_nacional.round(0))

print()
print("Top 5 departamentos, último año disponible ({}):".format(df_dpto.columns.max()))
print(df_dpto[df_dpto.columns.max()].nlargest(5).round(0))


Tabla exportada en: c:\Users\LENOVO\Desktop\Code\atlas-ambiental-colombia\examples\deforestacion-series-smbyc\deforestacion_departamentos.csv

Total nacional oficial por año (ha):
2013    120938.0
2014    140356.0
2015    124035.0
2016    178597.0
2017    219973.0
2018    197159.0
2019    158894.0
2020    171685.0
2021    174103.0
2022    123517.0
2023     79256.0
2024    113608.0
dtype: float64

Top 5 departamentos, último año disponible (2024):
departamento
META         27106.0
CAQUETÁ      25256.0
GUAVIARE     16943.0
ANTIOQUIA     7207.0
CHOCÓ         6336.0
Name: 2024, dtype: float64


In [15]:
fig = go.Figure()
fig.add_trace(go.Scatter(
    x=serie_nacional.index, y=serie_nacional.values,
    name="Total nacional", mode="lines+markers", line=dict(width=3, color="#333333"),
))
fig.add_trace(go.Scatter(
    x=serie_amazonia.index, y=serie_amazonia.values,
    name="Departamentos amazónicos", mode="lines+markers", line=dict(width=3, color="#2a7f3f"),
))
for nombre, fila in top_departamentos.iterrows():
    fig.add_trace(go.Scatter(
        x=fila.index, y=fila.values, name=str(nombre).title(),
        mode="lines", line=dict(width=1.5, dash="dot"),
    ))
fig.update_layout(
    title=dict(text="Deforestación anual en Colombia — cifras oficiales SMByC", x=0.5),
    xaxis=dict(title="Año", tickmode="linear", dtick=1),
    yaxis=dict(title="Hectáreas", tickformat=",.0f", gridcolor="#e0e0e0"),
    width=950, height=560,
    plot_bgcolor="white", paper_bgcolor="white",
    hovermode="x unified",
)
fig.show()
